# Prompt Engineering: Meal Plan Eval

Self-contained notebook for the prompt engineering lessons of *Building with the Claude API*.

**Run order:** cells 1-3 once per session (setup, helpers, evaluator), cell 4 once to create `dataset.json`, then iterate on the `run_prompt` cells.

After a kernel restart, use **Run All** or **Run Above** to rebuild the session.

## 1. Setup

In [2]:
import json
import re
from concurrent.futures import ThreadPoolExecutor
from statistics import mean

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()          # reads ANTHROPIC_API_KEY from .env
client = Anthropic()
MODEL = "claude-haiku-4-5"

## 2. Chat helpers
The small wrappers built in the *Making a request* and *Multi-turn conversations* lessons.

In [3]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send the conversation to Claude and return the reply text."""
    params = {
        "model": MODEL,
        "max_tokens": 2000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text


def extract_json(text):
    """Pull JSON out of a reply, whether or not it is wrapped in ```json fences."""
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    return json.loads(match.group(1) if match else text)

## 3. Prompt evaluator
Generates test cases, runs the prompt under test on each, and has Claude grade the outputs 1-10.

In [4]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    # Step A: have Claude invent test cases
    def generate_dataset(self, task_description, prompt_inputs_spec,
                         output_file="dataset.json", num_cases=3):
        spec = "\n".join(f'- "{k}": {v}' for k, v in prompt_inputs_spec.items())
        prompt = f"""Generate {num_cases} diverse, realistic test cases for evaluating a prompt.

<task_description>
{task_description}
</task_description>

Each test case supplies these inputs to the prompt:
{spec}

Return ONLY a JSON array. Each element is an object with:
- "prompt_inputs": an object with exactly the input keys listed above
- "solution_criteria": a list of 2-4 specific things a good output must do for this case
"""
        messages = []
        add_user_message(messages, prompt)
        dataset = extract_json(chat(messages))

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(dataset, f, indent=2)
        print(f"Wrote {len(dataset)} test cases to {output_file}")
        return dataset

    # Step B: have Claude grade one output
    def _grade(self, test_case, output, extra_criteria):
        criteria = "\n".join(f"- {c}" for c in test_case.get("solution_criteria", []))
        extra = f"\n<extra_criteria>\n{extra_criteria}\n</extra_criteria>\n" if extra_criteria else ""

        prompt = f"""You are an expert reviewer grading an AI-generated output.

<inputs>
{json.dumps(test_case["prompt_inputs"], indent=2)}
</inputs>

<output_to_grade>
{output}
</output_to_grade>

<solution_criteria>
{criteria}
</solution_criteria>
{extra}
Score the output from 1 (fails badly) to 10 (fully meets every criterion).
Return ONLY a JSON object with keys:
"strengths" (list), "weaknesses" (list), "reasoning" (string), "score" (integer 1-10)
"""
        messages = []
        add_user_message(messages, prompt)
        return extract_json(chat(messages, temperature=0.0))

    # Step C: run the prompt under test, then grade it
    def _run_case(self, run_prompt_function, test_case, extra_criteria):
        output = run_prompt_function(test_case["prompt_inputs"])
        grade = self._grade(test_case, output, extra_criteria)
        return {
            "test_case": test_case,
            "output": output,
            "score": grade["score"],
            "reasoning": grade["reasoning"],
            "weaknesses": grade.get("weaknesses", []),
        }

    def run_evaluation(self, run_prompt_function, dataset_file,
                       extra_criteria=None, output_file="results.json"):
        with open(dataset_file, encoding="utf-8") as f:
            dataset = json.load(f)

        # Run test cases in parallel, capped to avoid rate-limit errors
        with ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as pool:
            results = list(pool.map(
                lambda tc: self._run_case(run_prompt_function, tc, extra_criteria),
                dataset,
            ))

        avg = mean(r["score"] for r in results)
        print(f"Average score: {avg:.1f} / 10")
        for i, r in enumerate(results, 1):
            print(f"  Case {i}: {r['score']}/10 - {r['reasoning'][:120]}")

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)
        return results


evaluator = PromptEvaluator(max_concurrent_tasks=3)

## 4. Generate the test dataset
Run once. Re-running creates new test cases, which makes scores harder to compare across prompt versions.

In [5]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    output_file="dataset.json",
    num_cases=3,
)

Wrote 3 test cases to dataset.json


## 5. Shared grading criteria

In [6]:
EXTRA_CRITERIA = """
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""

## 6. Baseline prompt (deliberately naive)

In [7]:
def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


baseline_results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    output_file="results_baseline.json",
)

Average score: 4.3 / 10
  Case 1: 4/10 - The output provides solid nutritional guidance and correctly calculated targets that meet the caloric and macronutrient 
  Case 2: 6/10 - The output demonstrates strong nutritional knowledge and correctly addresses the solution criteria regarding vegetarian 
  Case 3: 3/10 - The output provides a solid foundational framework for a gluten-free, dairy-free weight loss plan with correct calculati


## 7. Technique: Being clear and direct

In [8]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


clear_direct_results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria=EXTRA_CRITERIA,
    output_file="results_clear_direct.json",
)

Average score: 7.7 / 10
  Case 1: 9/10 - The output successfully meets all solution criteria and extra criteria. It provides a comprehensive, well-structured mea
  Case 2: 8/10 - The output successfully meets all core solution criteria: it maintains strict vegetarianism with excellent protein diver
  Case 3: 6/10 - The output demonstrates strong technical execution in meal planning with excellent attention to dietary restrictions, pr


## Next techniques
Copy the cell above for each new lesson (being specific, XML tags, examples), edit the prompt, and give the results a new `output_file` name so you can compare runs.